# Unified Image Studio — MOSS + Anima
One ComfyUI server for both families, designed for Colab Free (T4 GPU, ~15 GiB usable VRAM and ~12.7 GiB system RAM). Choose **one** model, download only its components, and generate inside the notebook. No browser UI required.

**Quick start:** choose GPU → run sections 1–5 → optionally upload reference or LoRAs → set generation controls → generate. For switching models, pause any remote clients, run section 3 again with the new choice, then section 4 and the LoRA panel/configuration cells. A switch unloads the previous process; files remain cached on disk. No full runtime reset is required. Never switch while a job is running.

Anima Turbo v1.1 is the lightweight default; MOSS uses the same INT8 + W4A8 checkpoint configuration as the previously working MOSS notebook. Reference Adapter is Anima-only; Base v1.0 is the validated target. Model weights and LoRAs have their own licenses.

Authentication is optional and off by default. Google restricts remote proxy connections on managed Colab and certain external-UI uses on Free; use the tunnel only on a host that permits it. This notebook does not provision GPU runtimes, defeat idle limits, or implement the separate Spindle extension.

Sources: [Anima model card](https://huggingface.co/circlestone-labs/Anima), [MOSS ComfyUI weights](https://huggingface.co/Comfy-Org/MOSS-Image), [Anima Reference Adapter](https://github.com/ArcAnima/ComfyUI-Anima-Ref-Adapter), [Colab FAQ](https://research.google.com/colaboratory/faq.html), [cloudflared](https://github.com/cloudflare/cloudflared).

## 1 · GPU and workspace
Use **Runtime → Change runtime type → T4 GPU**. Not CPU or TPU. This check avoids importing PyTorch into the notebook kernel to conserve RAM.

In [ ]:
import os, sys, time, subprocess, shutil, json, random, io, gc
from pathlib import Path
import requests, psutil
COMFY = Path('/content/Unified_ComfyUI')
MODELS = COMFY / 'models'
BASE_URL = 'http://127.0.0.1:8188'
LOG = Path('/content/unified_comfyui.log')
ADAPTER = COMFY / 'custom_nodes' / 'ComfyUI-Anima-Ref-Adapter'
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True).strip())
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError('Select an NVIDIA GPU runtime (T4 or better).') from e

def resources():
    ram=psutil.virtual_memory()
    print(f'System available RAM: {ram.available/1024**3:.2f} GiB / {ram.total/1024**3:.2f} GiB total')
    subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader'])
    print(f'Local disk free: {shutil.disk_usage("/content").free/1024**3:.1f} GiB')
resources()

## 2 · Install ComfyUI and configure storage
Drive caching is optional and off by default. MOSS alone requires about 13 GB of weight files; caching multiple checkpoints can exceed a free Drive allocation. Anima and MOSS share one server but not model components. Leave updates off after a working install unless needed.

In [ ]:
# @title Installation and storage
USE_DRIVE = False # @param {type:"boolean"}
UPDATE_COMFYUI = False # @param {type:"boolean"}
INSTALL_ANIMA_ADAPTER = True # @param {type:"boolean"}
CACHE = Path('/content/unified_model_cache')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE=Path('/content/drive/MyDrive/Unified_Image_Assets')
CACHE.mkdir(parents=True,exist_ok=True)
# Use /content for HF transient cache even when Drive persistence is enabled.
os.environ.setdefault('HF_HOME','/content/unified_hf_cache')
def run_cmd(*args,cwd=None):
    subprocess.run([str(x) for x in args],cwd=cwd,check=True)
if not (COMFY/'main.py').is_file():
    run_cmd('git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',COMFY)
elif UPDATE_COMFYUI:
    run_cmd('git','pull','--ff-only',cwd=COMFY)
for category in ('diffusion_models','text_encoders','vae','loras'):
    (MODELS/category).mkdir(parents=True,exist_ok=True)
run_cmd(sys.executable,'-m','pip','install','-q','-r',COMFY/'requirements.txt')
run_cmd(sys.executable,'-m','pip','install','-q','huggingface_hub>=0.25','requests','pillow','ipywidgets','safetensors','psutil')
if INSTALL_ANIMA_ADAPTER and not ADAPTER.exists():
    run_cmd('git','clone','--depth','1','https://github.com/ArcAnima/ComfyUI-Anima-Ref-Adapter.git',ADAPTER)
print('Cache:',CACHE)
print('ComfyUI revision:', subprocess.check_output(['git','rev-parse','--short','HEAD'],cwd=COMFY,text=True).strip())

## 3 · Select and prepare a model
Only the selected checkpoint, encoder, and VAE are installed. Anima variants reuse their encoder/VAE. Model changes stop the owned ComfyUI server after checking its queue; **pause remote clients before switching**. Rerun section 4 to start the new session, then sections 5–9 as applicable. Existing weights stay on disk, not loaded in RAM. Do not edit the model name in the generation cell.

In [ ]:
# @title Model selection (rerun to switch)
MODEL_CHOICE = 'MOSS SFT INT8' # @param ['Anima Turbo v1.1','Anima Aesthetic v1.1','Anima Base v1.0','Anima Turbo v1.0','Anima Aesthetic v1.0','Anima Aesthetic v1.0b','MOSS SFT INT8','MOSS Base INT8']
DOWNLOAD_ADAPTER_WEIGHTS = False # @param {type:"boolean"}
from huggingface_hub import hf_hub_download
from safetensors import safe_open

ANIMA_REPO='circlestone-labs/Anima'
MOSS_REPO='Comfy-Org/MOSS-Image'
def make_profile(choice):
    if choice.startswith('MOSS '):
        flavor='sft' if 'SFT' in choice else 'base'
        return dict(family='moss',repo=MOSS_REPO,root='',checkpoint=f'moss_image_{flavor}_int8_convrot.safetensors',encoder='qwen3.5_9b_w4a8_gptq.safetensors',vae='flux2-vae_small.safetensors',clip_type='moss_image',latent='EmptyFlux2LatentImage',steps=25,cfg=2.5,sampler='er_sde',scheduler='simple')
    names={'Anima Base v1.0':'anima-base-v1.0.safetensors','Anima Aesthetic v1.0':'anima-aesthetic-v1.0.safetensors','Anima Aesthetic v1.0b':'anima-aesthetic-v1.0b.safetensors','Anima Aesthetic v1.1':'anima-aesthetic-v1.1.safetensors','Anima Turbo v1.0':'anima-turbo-v1.0.safetensors','Anima Turbo v1.1':'anima-turbo-v1.1.safetensors'}
    turbo='Turbo' in choice
    return dict(family='anima',repo=ANIMA_REPO,root='split_files/',checkpoint=names[choice],encoder='qwen_3_06b_base.safetensors',vae='qwen_image_vae.safetensors',clip_type='stable_diffusion',latent='EmptyLatentImage',steps=10 if turbo else 30,cfg=1.0 if turbo else 4.5,sampler='er_sde',scheduler='simple')

def comfy_ready():
    try: return requests.get(BASE_URL+'/system_stats',timeout=3).ok
    except requests.RequestException: return False

def require_idle():
    r=requests.get(BASE_URL+'/queue',timeout=10);r.raise_for_status();q=r.json()
    if q.get('queue_running') or q.get('queue_pending'):
        raise RuntimeError('ComfyUI has active or queued jobs. Wait until finished before changing models or stopping.')

def stop_owned_server():
    proc=globals().get('comfy_proc')
    if proc is None or proc.poll() is not None:
        if comfy_ready(): raise RuntimeError('Port 8188 is occupied by an unmanaged ComfyUI process. Stop the old notebook server first.')
        return
    if comfy_ready(): require_idle()
    proc.terminate()
    try: proc.wait(timeout=30)
    except subprocess.TimeoutExpired: proc.kill();proc.wait(timeout=15)
    print('Previous ComfyUI stopped; cached model RAM/VRAM released.')

p=make_profile(MODEL_CHOICE)
if comfy_ready() and (globals().get('RUNNING_CHOICE')!=MODEL_CHOICE or globals().get('ACTIVE_PROFILE')!=p):
    stop_owned_server()
ACTIVE_PROFILE=None  # Set only when all requested assets are ready.
LORA_CONFIG=[]; LORA_VALIDATED_FAMILY=None; LORA_VALIDATED_SIGNATURE=None
LORA_DIR=MODELS/'loras'/p['family'];LORA_DIR.mkdir(parents=True,exist_ok=True)
LORA_CACHE=CACHE/'loras'/p['family'];LORA_CACHE.mkdir(parents=True,exist_ok=True)

def acquire_weight(repo, repo_path, category):
    name=Path(repo_path).name
    dest=MODELS/category/name
    if dest.is_file() and dest.stat().st_size>100000:
        print('Installed:',name);return dest
    # local_dir avoids the extra full Hugging Face snapshot copy.
    storage=CACHE/'weights'/repo.replace('/','--')
    cached=Path(hf_hub_download(repo_id=repo,filename=repo_path,local_dir=storage))
    if dest.exists():dest.unlink()
    try: os.link(cached,dest)
    except OSError:
        if shutil.disk_usage(dest.parent).free < cached.stat().st_size + 512*1024**2:
            raise OSError('Not enough local disk space to stage model weights.')
        shutil.copy2(cached,dest)
    print('Ready:',name,f'({dest.stat().st_size/1024**3:.2f} GiB)')
    return dest

for category,key in [('diffusion_models','checkpoint'),('text_encoders','encoder'),('vae','vae')]:
    acquire_weight(p['repo'],p['root']+category+'/'+p[key],category)
if p['family']=='anima' and DOWNLOAD_ADAPTER_WEIGHTS:
    acquire_weight('ElectronGirl/anima-ref-adapter','anima_ref_adapter_v2_r64.safetensors','loras')
# Restore only this family's LoRAs. The adapter has a separate root filename.
for src in LORA_CACHE.glob('*.safetensors'):
    dst=LORA_DIR/src.name
    if USE_DRIVE and not dst.exists():
        shutil.copy2(src,dst)
ACTIVE_PROFILE=p
print('Selected:',MODEL_CHOICE,'| default:',p['steps'],'steps, CFG',p['cfg'])
resources()

## 4 · Start ComfyUI (one process)
Low VRAM is the tested default. Switching memory mode restarts ComfyUI only. On MOSS, system RAM may be the bottleneck; normal VRAM is optional and can cause GPU OOM. Pause remote clients and wait for the queue to clear before changing modes.

In [ ]:

# @title Launch settings
VRAM_MODE = 'normalvram' # @param ['lowvram','normalvram']

if ACTIVE_PROFILE is None:
    raise RuntimeError('Prepare a model in section 3.')

# Restart ComfyUI if the model or VRAM mode has changed
if comfy_ready() and (
    globals().get('RUNNING_CHOICE') != MODEL_CHOICE
    or globals().get('RUNNING_VRAM') != VRAM_MODE
):
    stop_owned_server()

if not comfy_ready():
    old = globals().get('comfy_proc')

    if old is not None and old.poll() is None:
        stop_owned_server()

    with LOG.open('a', buffering=1) as log_handle:

        launch_args = [
            sys.executable,
            'main.py',
            '--listen', '127.0.0.1',
            '--port', '8188'
        ]

        # Normal VRAM is the default in newer ComfyUI versions.
        # Only append a flag when using low VRAM mode.
        if VRAM_MODE == 'lowvram':
            launch_args.append('--lowvram')

        print('Launching ComfyUI:', ' '.join(launch_args))

        comfy_proc = subprocess.Popen(
            launch_args,
            cwd=COMFY,
            stdout=log_handle,
            stderr=subprocess.STDOUT
        )

    # Wait for ComfyUI to become available
    for _ in range(150):
        if comfy_ready() or comfy_proc.poll() is not None:
            break
        time.sleep(2)

    if not comfy_ready():
        print(
            LOG.read_text(errors='replace')[-7000:]
            if LOG.exists() else 'No log'
        )
        raise RuntimeError('ComfyUI failed to start.')

RUNNING_CHOICE = MODEL_CHOICE
RUNNING_VRAM = VRAM_MODE

print('ComfyUI ready:', BASE_URL, '|', RUNNING_CHOICE)
print('VRAM mode:', RUNNING_VRAM)


## 5 · Shared API helpers
The API graph is built for the selected family, including the correct latent format, encoder, and VAE.

In [ ]:
import io, json, math, random, shutil, time
from pathlib import Path

import requests
from PIL import Image
from IPython.display import display, Markdown

BASE_URL = 'http://127.0.0.1:8188'
def comfy_get(path, **kwargs):
    r = requests.get(BASE_URL + path, **kwargs)
    r.raise_for_status()
    return r.json()

def comfy_post(path, **kwargs):
    r = requests.post(BASE_URL + path, **kwargs)
    r.raise_for_status()
    return r.json()

def assert_comfy_running():
    r = requests.get(BASE_URL + '/system_stats', timeout=10)
    r.raise_for_status()
    return r.json()

def object_info():
    return comfy_get('/object_info', timeout=30)

def verify_required_nodes():
    info=object_info(); p=ACTIVE_PROFILE
    required=['UNETLoader','CLIPLoader','VAELoader','CLIPTextEncode',p['latent'],'KSampler','VAEDecode','SaveImage','LoraLoaderModelOnly']
    missing=[n for n in required if n not in info]
    if missing: raise RuntimeError('Missing nodes: '+str(missing))
    if p['clip_type'] not in info['CLIPLoader']['input']['required']['type'][0]: raise RuntimeError('Update ComfyUI: selected encoder type missing.')
    print('Native nodes verified for',MODEL_CHOICE)

def build_workflow(profile, positive, negative, width=768,height=768,steps=None,cfg=None,seed=-1,sampler=None,scheduler=None,loras=(),loader='LoraLoaderModelOnly',clip_device='default',use_reference=False,reference_filename=None,allow_experimental=False,ref_guidance=4.5,adapter_strength=1.0,ref_end=0.6,ref_megapixels=0.5,prefix='unified/image'):
    if min(width,height)<256 or width%16 or height%16: raise ValueError('Dimensions must be >=256 and multiples of 16.')
    if not positive.strip():raise ValueError('Enter a prompt.')
    if steps is not None and steps<1:raise ValueError('Steps must be positive.')
    if cfg is not None and cfg<0:raise ValueError('CFG must be nonnegative.')
    if use_reference:
        if profile['family']!='anima':raise ValueError('Anima Reference Adapter is not compatible with MOSS.')
        if profile['checkpoint']!='anima-base-v1.0.safetensors' and not allow_experimental:raise ValueError('Reference Adapter is trained on Anima Base v1.0. Experimental override required for other variants.')
        if not reference_filename:raise ValueError('Upload reference image first.')
    seed=random.randrange(2**63) if seed<0 else int(seed)
    g={}
    def node(i,kind,**inputs):
        g[str(i)]={'class_type':kind,'inputs':inputs};return [str(i),0]
    m=node(1,'UNETLoader',unet_name=profile['checkpoint'],weight_dtype='default')
    clip=node(2,'CLIPLoader',clip_name=profile['encoder'],type=profile['clip_type'],device=clip_device)
    v=node(3,'VAELoader',vae_name=profile['vae'])
    for i,(name,weight) in enumerate(loras,100):
        if float(weight)!=0:m=node(i,loader,model=m,lora_name=name,strength_model=float(weight))
    if use_reference:
        image=node(10,'LoadImage',image=reference_filename)
        m=node(11,'AnimaRefAdapter',model=m,vae=v,image=image,lora_name='anima_ref_adapter_v2_r64.safetensors',ref_guidance=float(ref_guidance),correction='apg',strength=float(adapter_strength),ref_end=float(ref_end),ref_megapixels=float(ref_megapixels))
    pos=node(4,'CLIPTextEncode',clip=clip,text=positive)
    neg=node(5,'CLIPTextEncode',clip=clip,text=negative)
    latent=node(6,profile['latent'],width=int(width),height=int(height),batch_size=1)
    sampled=node(7,'KSampler',model=m,positive=pos,negative=neg,latent_image=latent,seed=seed,steps=int(profile['steps'] if steps is None else steps),cfg=float(profile['cfg'] if cfg is None else cfg),sampler_name=sampler or profile['sampler'],scheduler=scheduler or profile['scheduler'],denoise=1.0)
    decoded=node(8,'VAEDecode',samples=sampled,vae=v)
    node(9,'SaveImage',images=decoded,filename_prefix=prefix)
    return g,seed

def queue_prompt(prompt):
    try:
        result = comfy_post('/prompt', json={'prompt': prompt}, timeout=120)
    except requests.HTTPError as exc:
        raise RuntimeError('ComfyUI rejected workflow: '+exc.response.text[:2200]) from exc
    return result['prompt_id']

def wait_for_completion(prompt_id, timeout=1800, poll=2):
    started = time.time()
    while time.time() - started < timeout:
        history = comfy_get(f'/history/{prompt_id}', timeout=60)
        if prompt_id in history:
            return history[prompt_id]
        time.sleep(poll)
    raise TimeoutError(f'Prompt {prompt_id} did not finish within {timeout} seconds.')

def extract_images_from_history(history_entry):
    images = []
    outputs = history_entry.get('outputs', {})
    for node_id, node_output in outputs.items():
        for img in node_output.get('images', []):
            images.append(img)
    return images

def fetch_image(image_meta):
    params = {
        'filename': image_meta['filename'],
        'subfolder': image_meta.get('subfolder', ''),
        'type': image_meta.get('type', 'output'),
    }
    r = requests.get(BASE_URL + '/view', params=params, timeout=120)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert('RGB')

def maybe_copy_output_to_cache(image_meta):
    if not USE_DRIVE:
        return None
    src = COMFY / 'output' / image_meta.get('subfolder', '') / image_meta['filename']
    dst_dir = CACHE / 'outputs' / ACTIVE_PROFILE['family']
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / image_meta['filename']
    if src.exists():
        shutil.copy2(src, dst)
        return dst
    return None

assert_comfy_running()
verify_required_nodes()

## 6 · Optional Anima reference image
Disabled by default. MOSS cannot use the ArcAnima adapter. For Anima Base v1.0, upload one reference here then enable the adapter in section 9. Other Anima versions require an experimental override.

In [ ]:
# @title Upload reference (optional)
UPLOAD_REFERENCE=False # @param {type:"boolean"}
REFERENCE_LOCAL_PATH=globals().get('REFERENCE_LOCAL_PATH')
if UPLOAD_REFERENCE:
    if ACTIVE_PROFILE['family']!='anima': raise ValueError('Select Anima for the Reference Adapter.')
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Upload exactly one reference image.')
    name,payload=next(iter(uploaded.items()))
    ref=Path('/content/unified_reference.png')
    with Image.open(io.BytesIO(payload)) as im:
        im.convert('RGB').save(ref)
    REFERENCE_LOCAL_PATH=str(ref)
    del uploaded,payload
    gc.collect()
    display(Image.open(ref))
else: print('Reference upload skipped.')

## 7 · Optional manual LoRA upload
For Civitai downloads that require browser authentication, download a compatible `.safetensors` file and upload it here. Use the exact filename in the next panel. Files are separated by model family; local uploads cannot be checked for architecture automatically. Use small files on Free Colab because uploads temporarily occupy notebook RAM.

In [ ]:
# @title Upload local LoRAs (optional)
UPLOAD_LOCAL_LORA=True # @param {type:"boolean"}
if UPLOAD_LOCAL_LORA:
    from google.colab import files
    uploaded=files.upload()
    for original,data in uploaded.items():
        name=Path(original).name
        if not name.lower().endswith('.safetensors'):print('Skipped:',name);continue
        dst=LORA_DIR/name; tmp=dst.with_suffix('.part')
        try:
            tmp.write_bytes(data)
            with safe_open(str(tmp),framework='np') as handle:
                if not list(handle.keys()):raise ValueError('Empty SafeTensors file')
            tmp.replace(dst)
            if USE_DRIVE:shutil.copy2(dst,LORA_CACHE/name)
            print('Installed:',name)
        finally:tmp.unlink(missing_ok=True)
    del uploaded
    if 'data' in globals():del data
    gc.collect()
    LORA_CONFIG=[];LORA_VALIDATED_SIGNATURE=None
else:print('Upload skipped.')

## 8 · Weighted Civitai LoRA manager
Paste ordinary Civitai model URLs (`.com` or `.red`), explicit version links, or direct Civitai download links. The manager finds a matching Anima or MOSS version if no version is specified, validates explicit versions, checks SafeTensors and hashes when available, and streams the file rather than storing its bytes in Python. Individual weights are preserved. Local filenames are supported. Civitai may still deny access; use the optional token or manual upload. Rerun this panel after model switches. Keep compatible LoRAs small and use only a few on MOSS. Anima variants share a family, but individual LoRAs may behave differently on Turbo/Aesthetic.

In [ ]:
import re, struct, hashlib, ipywidgets as widgets
from urllib.parse import urlparse, parse_qs
from IPython.display import display, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass

lora_rows=[]
LORA_CONFIG=[]
LORA_VALIDATED_FAMILY=None
LORA_VALIDATED_SIGNATURE=None
PANEL_FAMILY=ACTIVE_PROFILE['family']
lora_message=widgets.Output()
rows_box=widgets.VBox([])
status=widgets.HTML('<b>LoRAs:</b> none selected')
token_input=widgets.Password(description='Civitai token:',placeholder='Optional; never shared',style={'description_width':'100px'},layout=widgets.Layout(width='440px'))
add_btn=widgets.Button(description='Add LoRA',icon='plus')
apply_btn=widgets.Button(description='Download / validate',button_style='success',icon='download')

# Use the API for precise version detection and provenance, not page scraping.
CIVITAI_HOSTS={'civitai.com','www.civitai.com','civitai.red','www.civitai.red'}

def civitai_get_json(endpoint,headers):
    r=requests.get('https://civitai.com'+endpoint,headers=headers,timeout=60)
    if r.status_code in (401,403): raise PermissionError('Civitai metadata requires authorization or is restricted.')
    r.raise_for_status()
    return r.json()

def compatible_base(label):
    normalized=re.sub(r'[^a-z0-9]','',str(label).lower())
    return (normalized.startswith('mossimage') or normalized=='moss') if ACTIVE_PROFILE['family']=='moss' else normalized.startswith('anima')

def resolve_civitai(source,headers):
    parsed=urlparse(source)
    if parsed.scheme!='https' or parsed.hostname not in CIVITAI_HOSTS:
        raise ValueError('Use a Civitai HTTPS model page, version link, direct Civitai download URL, or installed filename.')
    q=parse_qs(parsed.query)
    version_id=(q.get('modelVersionId') or [None])[0]
    m=re.search(r'/api/download/models/(\d+)',parsed.path) or re.search(r'/model-versions/(\d+)',parsed.path)
    if m: version_id=m.group(1)
    if not version_id:
        match=re.search(r'/models/(\d+)',parsed.path)
        if not match: raise ValueError('Model URL missing /models/ID.')
        model=civitai_get_json(f'/api/v1/models/{match.group(1)}',headers)
        candidates=[v for v in model.get('modelVersions',[]) if compatible_base(v.get('baseModel'))]
        if not candidates:
            available=', '.join(f"{v.get('name')} ({v.get('baseModel')})" for v in model.get('modelVersions',[]))
            raise ValueError('No compatible version for the selected model family. Available: '+available[:700])
        # Civitai typically returns newest first. Prefer exact base model MOSS-Image.
        chosen=next((v for v in candidates if compatible_base(v.get('baseModel'))),candidates[0])
        version_id=str(chosen['id'])
        print('Automatically selected compatible version:',chosen.get('name'),version_id)
    version=civitai_get_json(f'/api/v1/model-versions/{int(version_id)}',headers)
    model=version.get('model',{})
    if model.get('type','').lower()!='lora':
        raise ValueError('Civitai resource type is not LoRA: '+str(model.get('type')))
    if not compatible_base(version.get('baseModel')):
        raise ValueError(f"Version {version_id} targets {version.get('baseModel')}, not the selected model family. Choose a compatible version.")
    files=[f for f in version.get('files',[]) if f.get('name','').lower().endswith('.safetensors')]
    if not files: raise ValueError('This version has no public .safetensors file.')
    wanted=(q.get('fileId') or [None])[0]
    selected=next((f for f in files if str(f.get('id'))==str(wanted)),None) if wanted else None
    if wanted and selected is None: raise ValueError('Requested fileId is not a SafeTensors file in this version.')
    selected=selected or next((f for f in files if f.get('primary')),files[0])
    url=selected.get('downloadUrl') or version.get('downloadUrl')
    if not url: raise ValueError('Civitai has no accessible download URL for this version.')
    print('LoRA:',model.get('name'), '| version:',version.get('name'),'| base:',version.get('baseModel'))
    return version,selected,url

def check_safetensors(path):
    with safe_open(str(path),framework='np') as handle:
        if not list(handle.keys()):raise ValueError('Empty SafeTensors file')

def download_lora(source,headers):
    if not source.startswith('https://'):
        name=Path(source).name
        if name.startswith('anima_ref_adapter'):raise ValueError('Use the dedicated Reference Adapter switch instead.')
        if source not in (name, ACTIVE_PROFILE['family']+'/'+name): raise ValueError('Select a local LoRA from the active family.')
        if not (LORA_DIR/name).is_file():
            raise FileNotFoundError('LoRA not installed: '+source)
        check_safetensors(LORA_DIR/name)
        return ACTIVE_PROFILE['family']+'/'+name
    version, selected, url = resolve_civitai(source,headers)
    # Prefix version+file IDs to avoid collisions across models with identical filenames.
    name=f"civitai_{version['id']}_{selected['id']}_{Path(selected['name']).name}"
    destination=LORA_DIR/name
    cached=LORA_CACHE/name
    cached.parent.mkdir(parents=True,exist_ok=True)
    if destination.is_file() and destination.stat().st_size>1024:
        check_safetensors(destination)
        print('Already installed:',name)
        return ACTIVE_PROFILE['family']+'/'+name
    if cached.is_file() and cached.stat().st_size>1024:
        shutil.copy2(cached,destination)
        check_safetensors(destination)
        print('Restored from cache:',name)
        return ACTIVE_PROFILE['family']+'/'+name
    temp=destination.with_name(destination.name+'.part')
    try:
        with requests.get(url,headers=headers if urlparse(url).hostname in CIVITAI_HOSTS else {},stream=True,timeout=180) as r:
            if r.status_code in (401,403):
                raise PermissionError('Civitai refused this file (401/403). Enter your Civitai token or download the compatible file in your browser and upload it manually.')
            r.raise_for_status()
            if 'text/html' in r.headers.get('Content-Type','').lower() or 'application/json' in r.headers.get('Content-Type','').lower():
                raise RuntimeError('Civitai returned an error page, not a model file.')
            with open(temp,'wb') as f:
                for chunk in r.iter_content(1024*1024):
                    if chunk: f.write(chunk)
        check_safetensors(temp)
        expected=(selected.get('hashes') or {}).get('SHA256')
        if expected and len(expected)==64:
            sha=hashlib.sha256()
            with open(temp,'rb') as f:
                for chunk in iter(lambda:f.read(4*1024*1024),b''): sha.update(chunk)
            if sha.hexdigest().lower()!=expected.lower(): raise ValueError('Downloaded file failed SHA256 verification.')
        temp.replace(destination)
        if USE_DRIVE: shutil.copy2(destination,cached)
        print('Installed:',name, f'({destination.stat().st_size/1048576:.1f} MiB)')
    finally:
        temp.unlink(missing_ok=True)
    return ACTIVE_PROFILE['family']+'/'+name

def invalidate(_=None):
    global LORA_CONFIG,LORA_VALIDATED_FAMILY,LORA_VALIDATED_SIGNATURE
    LORA_CONFIG=[]
    LORA_VALIDATED_FAMILY=None
    LORA_VALIDATED_SIGNATURE=None
    status.value='<b>Changed:</b> click Download / validate before generating.'

def render_rows():
    rows_box.children=tuple(widgets.HBox([r['source'],r['strength'],r['enabled'],r['remove']]) for r in lora_rows)

def add_row(source='',weight=0.8,enabled=True):
    field=widgets.Text(value=source,description='LoRA:',placeholder='Civitai page URL / version URL / filename',layout=widgets.Layout(width='65%'),style={'description_width':'50px'})
    strength=widgets.FloatText(value=weight,description='Weight:',layout=widgets.Layout(width='145px'),style={'description_width':'65px'})
    checked=widgets.Checkbox(value=enabled,description='Use',indent=False,layout=widgets.Layout(width='70px'))
    remove=widgets.Button(description='Remove',icon='trash')
    row={'source':field,'strength':strength,'enabled':checked,'remove':remove}
    lora_rows.append(row)
    def remove_it(_):
        lora_rows.remove(row);render_rows();invalidate()
    remove.on_click(remove_it)
    for w in (field,strength,checked):w.observe(invalidate,names='value')
    render_rows();invalidate()

def active_signature():
    return tuple((r['source'].value.strip(),float(r['strength'].value)) for r in lora_rows if r['enabled'].value and r['source'].value.strip())

def prepare_loras(_=None):
    global LORA_CONFIG,LORA_VALIDATED_FAMILY,LORA_VALIDATED_SIGNATURE
    LORA_CONFIG=[]
    LORA_VALIDATED_FAMILY=None
    LORA_VALIDATED_SIGNATURE=None
    with lora_message:
        clear_output(wait=True)
        active=[r for r in lora_rows if r['enabled'].value and r['source'].value.strip()]
        headers={'Authorization':'Bearer '+token_input.value.strip()} if token_input.value.strip() else {}
        prepared=[]
        try:
            if PANEL_FAMILY!=ACTIVE_PROFILE['family']:raise RuntimeError('Rerun the LoRA panel after switching model families.')
            if comfy_ready():require_idle()
            for r in active:
                weight=float(r['strength'].value)
                if not -5<=weight<=5: raise ValueError('LoRA strength outside -5..5')
                prepared.append((download_lora(r['source'].value.strip(),headers),weight))
            LORA_CONFIG=prepared
            LORA_VALIDATED_FAMILY=ACTIVE_PROFILE['family']
            LORA_VALIDATED_SIGNATURE=active_signature()
            status.value=f'<b>Ready:</b> {len(prepared)} LoRA(s)'
            print('Ready:',LORA_CONFIG)
        except Exception as e:
            status.value='<b>Failed:</b> inspect the error below'
            print(type(e).__name__+':',str(e).split(' for url:')[0])
        finally:
            token_input.value=''
            # Keep empty config; generation fails safely rather than using stale weights.

add_btn.on_click(lambda _:add_row())
apply_btn.on_click(prepare_loras)
add_row()
add_row(enabled=False)
display(widgets.VBox([widgets.HTML('<h3>Weighted '+ACTIVE_PROFILE['family'].upper()+' LoRAs — regular Civitai URLs</h3>'),rows_box,add_btn,token_input,apply_btn,status,lora_message]))

## 9 · Generation settings
Automatic presets follow the selected profile: MOSS 25 steps/CFG 2.5, Anima Base/Aesthetic 30/4.5, Turbo 10/1.0. Use override values only if desired. Resolution 768×768, batch size 1. For memory issues try 512×768; first generation after a switch is slower. Reference defaults use 0.5 MP and ref_end 0.6 to reduce cost; increase if needed.

In [ ]:
# @title Generation controls
POSITIVE_PROMPT='A cinematic 1990s anime illustration of a quiet cafe at dusk, warm light through the window, hand-painted backgrounds, rich shadows.' # @param {type:"string"}
NEGATIVE_PROMPT='diagonal streaks, speed lines, repeated stripes, banding, glitch texture, motion blur, distorted pattern,  worst quality, low quality, text, logo, bad hands, missing fingers, multiple limbs, bad anatomy, bobbles,' # @param {type:"string"}
WIDTH=1024 # @param {type:"integer"}
HEIGHT=1024 # @param {type:"integer"}
USE_MODEL_PRESET=True # @param {type:"boolean"}
STEPS=25 # @param {type:"integer"}
CFG=2.5 # @param {type:"number"}
SEED=-1 # @param {type:"integer"}
SAMPLER='er_sde' # @param ['er_sde','euler','dpmpp_2m']
SCHEDULER='simple' # @param ['simple','normal','beta']
CLIP_DEVICE='default' # @param ['default','cpu']
LORA_LOADER='LoraLoaderModelOnly' # @param ['LoraLoaderModelOnly','LoraLoaderBypassModelOnly']
USE_REFERENCE_ADAPTER=False # @param {type:"boolean"}
ALLOW_EXPERIMENTAL_ADAPTER=False # @param {type:"boolean"}
REF_GUIDANCE=4.5 # @param {type:"number"}
ADAPTER_STRENGTH=1.0 # @param {type:"number"}
REF_END=0.6 # @param {type:"number"}
REF_MEGAPIXELS=0.5 # @param {type:"number"}
if USE_REFERENCE_ADAPTER and ACTIVE_PROFILE['family']!='anima':raise ValueError('Disable Reference Adapter for MOSS.')
if USE_REFERENCE_ADAPTER and ACTIVE_PROFILE['checkpoint']!='anima-base-v1.0.safetensors' and not ALLOW_EXPERIMENTAL_ADAPTER:raise ValueError('Select Anima Base v1.0 or explicitly enable experimental adapter use.')
print('Active model:',MODEL_CHOICE,'| resolution:',WIDTH,'x',HEIGHT)
print('Steps / CFG:',(ACTIVE_PROFILE['steps'],ACTIVE_PROFILE['cfg']) if USE_MODEL_PRESET else (STEPS,CFG))

## 10 · Generate inline and export API workflow
The last successful generation graph is saved for Lumiverse. Keep installed filenames unchanged on the backend; family-specific LoRAs appear as `anima/...` or `moss/...` in the API graph. A remote client should import the correct model-specific workflow.

In [ ]:
assert_comfy_running()
if RUNNING_CHOICE!=MODEL_CHOICE or ACTIVE_PROFILE is None:raise RuntimeError('Prepare and start the selected model first.')
p=ACTIVE_PROFILE
if 'lora_rows' not in globals() or PANEL_FAMILY!=p['family']:raise RuntimeError('Run the LoRA manager for this family first.')
if active_signature() and (LORA_VALIDATED_FAMILY!=p['family'] or LORA_VALIDATED_SIGNATURE!=active_signature()):raise RuntimeError('Click Download / validate after selecting LoRAs.')
loras=list(LORA_CONFIG) if active_signature() else []
info=object_info()
if loras and LORA_LOADER not in info:raise RuntimeError('Selected LoRA loader unavailable; use LoraLoaderModelOnly.')
# Check installed assets and registered backend LoRA names before submitting.
for name,weight in loras:
    if not (MODELS/'loras'/name).is_file():raise FileNotFoundError(name)
ref_name=None
if USE_REFERENCE_ADAPTER:
    if p['family']!='anima':raise ValueError('Reference Adapter cannot be used with MOSS.')
    if 'AnimaRefAdapter' not in info:raise RuntimeError('Install adapter in section 2 and restart ComfyUI.')
    if not globals().get('REFERENCE_LOCAL_PATH') or not Path(REFERENCE_LOCAL_PATH).is_file():raise RuntimeError('Upload reference in section 6.')
    with open(REFERENCE_LOCAL_PATH,'rb') as f:
        result=comfy_post('/upload/image',files={'image':('unified_reference.png',f,'image/png')},data={'type':'input','overwrite':'true'},timeout=120)
    ref_name=(result.get('subfolder','')+'/' if result.get('subfolder') else '')+result['name']
workflow,actual_seed=build_workflow(p,POSITIVE_PROMPT,NEGATIVE_PROMPT,width=WIDTH,height=HEIGHT,steps=None if USE_MODEL_PRESET else STEPS,cfg=None if USE_MODEL_PRESET else CFG,seed=SEED,sampler=None if USE_MODEL_PRESET else SAMPLER,scheduler=None if USE_MODEL_PRESET else SCHEDULER,loras=loras,loader=LORA_LOADER,clip_device=CLIP_DEVICE,use_reference=USE_REFERENCE_ADAPTER,reference_filename=ref_name,allow_experimental=ALLOW_EXPERIMENTAL_ADAPTER,ref_guidance=REF_GUIDANCE,adapter_strength=ADAPTER_STRENGTH,ref_end=REF_END,ref_megapixels=REF_MEGAPIXELS,prefix=p['family']+'/image')
print('Submitting:',MODEL_CHOICE,'seed:',actual_seed,'LoRAs:',loras)
prompt_id=queue_prompt(workflow)
print('Prompt ID:',prompt_id)
record=wait_for_completion(prompt_id,timeout=3600)
if record.get('status',{}).get('status_str')=='error':raise RuntimeError(str(record['status'])[-4000:])
images=extract_images_from_history(record)
if not images:raise RuntimeError('No images returned. Inspect server log below.')
workflow_path=Path('/content/unified_workflow_api.json');workflow_path.write_text(json.dumps(workflow,indent=2))
family_path=Path('/content/'+p['family']+'_workflow_api.json');family_path.write_text(json.dumps(workflow,indent=2))
for item in images:
    display(fetch_image(item));print('Saved:',item['filename'])
    copied=maybe_copy_output_to_cache(item)
    if copied:print('Drive:',copied)
print('API workflow:',family_path)

In [ ]:
# @title Optional: download last successful workflow
DOWNLOAD_WORKFLOW=False # @param {type:"boolean"}
if DOWNLOAD_WORKFLOW:
    from google.colab import files
    path=Path('/content/unified_workflow_api.json')
    if path.is_file():files.download(str(path))
    else:print('Generate an image first.')
else:print('Workflow download skipped.')

## 11 · Optional authenticated tunnel
For hosts that permit remote tunneling. On managed Colab, external proxies are restricted; keep this off on Free. This is connectivity only, not GPU provisioning. ComfyUI remains bound to localhost; a small Caddy reverse proxy adds HTTP Basic Auth in front of it, and a free Cloudflare quick tunnel (`trycloudflare.com`, no account, no payment) publishes it. The tunnel URL is ephemeral — copy it into the Lumiverse Space secrets after each change. Native Lumiverse needs the implementation-side gateway; the credentials are prompted privately and never written into workflow JSON. A successful health check from this notebook does not prove reachability from the Space.


In [ ]:
# @title Optional authenticated tunnel (cloudflared quick tunnel)
ENABLE_TUNNEL = False # @param {type:"boolean"}
import sys, subprocess, time, secrets, re
from getpass import getpass
from pathlib import Path
import requests

if ENABLE_TUNNEL:
    if not comfy_ready(): raise RuntimeError('Start ComfyUI first.')
    CADDY = Path('/content/caddy')
    CF = Path('/content/cloudflared')
    if not CADDY.exists():
        subprocess.run(['bash','-c','curl -L -o /content/caddy.tgz https://github.com/caddyserver/caddy/releases/download/v2.8.4/caddy_2.8.4_linux_amd64.tar.gz && tar -xzf /content/caddy.tgz -C /content caddy && rm /content/caddy.tgz'],check=True)
        CADDY.chmod(0o755)
    if not CF.exists():
        subprocess.run(['bash','-c','curl -L -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],check=True)
        CF.chmod(0o755)
    for name in ('tunnel_caddy_proc','tunnel_cf_proc'):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None: proc.terminate(); proc.wait(timeout=15)
    username = input('HTTP Basic Auth username: ').strip()
    password = getpass('HTTP Basic Auth password: ').strip()
    if not username or ':' in username or not password: raise ValueError('Need a username (no colons) and a password.')
    hashed = subprocess.check_output([str(CADDY),'hash-password','--plaintext',password],text=True).strip()
    with open('/content/Caddyfile','w') as f:
        f.write(f':8189 {{\n\tbasic_auth {{\n\t\t{username} {hashed}\n\t}}\n\treverse_proxy 127.0.0.1:8188\n}}\n')
    tunnel_caddy_proc = subprocess.Popen([str(CADDY),'run','--config','/content/Caddyfile'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    time.sleep(2)
    r = requests.get('http://127.0.0.1:8189/system_stats',auth=(username,password),timeout=10)
    assert r.ok and 'devices' in r.json(), 'Local proxy health check failed.'
    tunnel_cf_proc = subprocess.Popen([str(CF),'tunnel','--url','http://127.0.0.1:8189'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    TUNNEL_URL = None
    for line in tunnel_cf_proc.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m: TUNNEL_URL = m.group(0); break
    print('Tunnel online:', TUNNEL_URL)
    print('Basic Auth username:', username)
    print('Basic Auth password: (the one you just typed; not printed)')
    print('Set these in the HF Space secrets and RESTART the Space:')
    print('  GATEWAY_UPSTREAM=' + TUNNEL_URL)
    print('  GATEWAY_BASIC_AUTH=' + username + ':<password you typed>')
    # Note: the Colab runtime cannot reliably curl its own *.trycloudflare.com
    # hostname (egress hairpin restriction). Health is verified from outside
    # (the Lumiverse gateway), not from inside the runtime.
    print('Tunnel registered. Verify it with the Lumiverse gateway, not from this runtime.')
else: print('Tunnel disabled. Inline generation works without it.')


## 12 · Diagnostics, cache release, and shutdown
Memory reported while idle includes cached model weights. Unload releases cache but the next job must reload. Only use these controls when no jobs are active, and pause outside clients first. Stopping ComfyUI does not delete the Colab VM; use Runtime → Disconnect and delete runtime when done.

In [ ]:
# @title Diagnostics / safe cleanup
UNLOAD_CACHED_MODELS=False # @param {type:"boolean"}
CLOSE_TUNNEL=False # @param {type:"boolean"}
STOP_COMFYUI=False # @param {type:"boolean"}
if CLOSE_TUNNEL:
    for name in ('tunnel_cf_proc','tunnel_caddy_proc'):
        proc = globals().get(name)
        if proc is not None and proc.poll() is None: proc.terminate(); proc.wait(timeout=15)
    print('Tunnel closed.')
if UNLOAD_CACHED_MODELS and comfy_ready():
    require_idle()
    r=requests.post(BASE_URL+'/free',json={'unload_models':True,'free_memory':True},timeout=30);r.raise_for_status()
    time.sleep(5)
if STOP_COMFYUI:stop_owned_server()
gc.collect();resources()
print('ComfyUI alive:',comfy_ready())
print('Last log lines:')
print(LOG.read_text(errors='replace')[-5000:] if LOG.exists() else 'No logs.')